In [8]:
import pandas as pd

file_path = r"..\..\data\raw\ev_charging_dataset.xlsx"

stations = pd.read_excel(file_path, sheet_name="stations")
chargers = pd.read_excel(file_path, sheet_name="chargers")
sessions = pd.read_excel(file_path, sheet_name="sessions")

print("Stations:", stations.shape)
print("Chargers:", chargers.shape)
print("Sessions:", sessions.shape)

print("\n--- Charger columns ---")
print(chargers.columns.tolist())

print("\n--- Session columns ---")
print(sessions.columns.tolist())

print("\n--- Charger data sample ---")
display(chargers.head())

print("\n--- Session data sample ---")
display(sessions.head())


Stations: (33, 5)
Chargers: (199, 5)
Sessions: (294024, 15)

--- Charger columns ---
['charger_id', 'station_id', 'charger_type', 'max_power_kW', 'installation_date']

--- Session columns ---
['session_id', 'station_id', 'charger_id', 'charger_type', 'start_timestamp', 'end_timestamp', 'session_duration_minutes', 'energy_kwh', 'price_per_kwh', 'total_cost', 'payment_method', 'user_type', 'customer_id', 'station_region', 'year']

--- Charger data sample ---


,charger_id,station_id,charger_type,max_power_kW,installation_date
0,STN_001_CH_01,STN_001,DC_50kW,50,2022-06-07
1,STN_001_CH_02,STN_001,DC_150kW,150,2022-12-22
2,STN_001_CH_03,STN_001,DC_150kW,150,2022-10-21
3,STN_001_CH_04,STN_001,DC_50kW,50,2022-12-08
4,STN_002_CH_01,STN_002,DC_50kW,50,2022-07-09



--- Session data sample ---


,session_id,station_id,charger_id,charger_type,start_timestamp,end_timestamp,session_duration_minutes,energy_kwh,price_per_kwh,total_cost,payment_method,user_type,customer_id,station_region,year
0,S_00000001,STN_001,STN_001_CH_02,DC_150kW,2022-01-01 08:30:00,2022-01-01 08:56:00,26,56.9,0.47,26.74,app,fleet,CUST_00134,North West,2022
1,S_00000002,STN_001,STN_001_CH_02,DC_150kW,2022-01-01 09:20:00,2022-01-01 09:41:00,21,40.7,0.46,18.72,app,delivery,CUST_00195,North West,2022
2,S_00000003,STN_001,STN_001_CH_04,DC_50kW,2022-01-01 09:00:00,2022-01-01 09:48:00,48,30.9,0.47,14.52,rfid,public,CUST_01935,North West,2022
3,S_00000004,STN_001,STN_001_CH_01,DC_50kW,2022-01-01 23:20:00,2022-01-02 00:01:00,41,28.4,0.45,12.78,rfid,public,CUST_00036,North West,2022
4,S_00000005,STN_001,STN_001_CH_01,DC_50kW,2022-01-01 16:10:00,2022-01-01 17:00:00,50,31.7,0.56,17.75,rfid,taxi,CUST_00011,North West,2022


In [10]:

# Estimate average delivered power during each session
sessions["avg_delivered_power_kW"] = (
    sessions["energy_kwh"]
    / (sessions["session_duration_minutes"] / 60)
)

# Summarise performance for each charger
charger_summary = (
    sessions.groupby("charger_id")
    .agg(
        total_sessions=("session_id", "count"),
        avg_delivered_power_kW=("avg_delivered_power_kW", "mean"),
        avg_energy_kwh=("energy_kwh", "mean"),
        avg_duration_minutes=("session_duration_minutes", "mean")
    )
    .reset_index()
)

# Add each charger's rated power
charger_summary = charger_summary.merge(
    chargers[["charger_id", "max_power_kW", "installation_date"]],
    on="charger_id",
    how="left"
)

charger_summary["capacity_utilisation_pct"] = (
    charger_summary["avg_delivered_power_kW"]
    / charger_summary["max_power_kW"]
    * 100
)

display(charger_summary.head())
print("Number of chargers:", charger_summary["charger_id"].nunique())
print("\nSummary statistics:")
display(charger_summary.describe().round(2))


,charger_id,total_sessions,avg_delivered_power_kW,avg_energy_kwh,avg_duration_minutes,max_power_kW,installation_date,capacity_utilisation_pct
0,STN_001_CH_01,3731,40.018430,33.046234,49.546770,50,2022-06-07,80.036859
1,STN_001_CH_02,3706,119.962211,48.871398,24.444145,150,2022-12-22,79.974808
2,STN_001_CH_03,3877,119.952877,48.898839,24.460408,150,2022-10-21,79.968585
3,STN_001_CH_04,3848,40.018214,32.993217,49.466736,50,2022-12-08,80.036428
4,STN_002_CH_01,3084,40.001531,32.979215,49.467899,50,2022-07-09,80.003063


Number of chargers: 199

Summary statistics:


,total_sessions,avg_delivered_power_kW,avg_energy_kwh,avg_duration_minutes,max_power_kW,installation_date,capacity_utilisation_pct
count,199.00,199.00,199.00,199.00,199.00,199,199.00
mean,1477.51,101.68,46.00,35.54,128.64,2023-10-23 04:06:01.809045248,79.56
min,586.00,39.78,32.70,19.33,50.00,2022-01-04 00:00:00,77.40
25%,743.50,39.98,32.99,24.46,50.00,2023-02-11 12:00:00,79.75
50%,1258.00,119.59,48.76,24.58,150.00,2023-12-16 00:00:00,79.92
75%,1879.50,120.11,49.04,49.47,150.00,2024-07-12 12:00:00,80.02
max,3877.00,234.16,75.77,49.82,300.00,2024-12-28 00:00:00,80.28
std,897.04,70.71,15.45,13.46,91.20,NaN,0.86


In [11]:

sessions["start_timestamp"] = pd.to_datetime(
    sessions["start_timestamp"]
)

sessions["month"] = (
    sessions["start_timestamp"].dt.to_period("M")
)

sessions["avg_delivered_power_kW"] = (
    sessions["energy_kwh"]
    / (sessions["session_duration_minutes"] / 60)
)

monthly_charger = (
    sessions.groupby(["charger_id", "month"])
    .agg(
        sessions=("session_id", "count"),
        avg_power_kW=("avg_delivered_power_kW", "mean"),
        avg_energy_kwh=("energy_kwh", "mean"),
        avg_duration_minutes=(
            "session_duration_minutes", "mean"
        )
    )
    .reset_index()
)

monthly_charger = monthly_charger.merge(
    chargers[["charger_id", "max_power_kW"]],
    on="charger_id",
    how="left"
)

monthly_charger["capacity_utilisation_pct"] = (
    monthly_charger["avg_power_kW"]
    / monthly_charger["max_power_kW"]
    * 100
)

print("Monthly charger records:", len(monthly_charger))
print("Unique chargers:", monthly_charger["charger_id"].nunique())
print("Months covered:", monthly_charger["month"].nunique())

display(monthly_charger.head())


Monthly charger records: 4128
Unique chargers: 199
Months covered: 36


,charger_id,month,sessions,avg_power_kW,avg_energy_kwh,avg_duration_minutes,max_power_kW,capacity_utilisation_pct
0,STN_001_CH_01,2022-01,123,39.832892,32.941463,49.601626,50,79.665783
1,STN_001_CH_01,2022-02,81,40.404324,33.472840,49.728395,50,80.808648
2,STN_001_CH_01,2022-03,111,39.884607,33.384685,50.216216,50,79.769214
3,STN_001_CH_01,2022-04,101,39.806824,33.104950,49.861386,50,79.613648
4,STN_001_CH_01,2022-05,116,40.127238,33.573276,50.215517,50,80.254476


In [12]:

# Attach installation dates to monthly records
monthly_check = monthly_charger.merge(
    chargers[["charger_id", "installation_date"]],
    on="charger_id",
    how="left"
)

monthly_check["month_start"] = (
    monthly_check["month"].dt.to_timestamp()
)

monthly_check["before_installation"] = (
    monthly_check["month_start"]
    < monthly_check["installation_date"]
)

print("Monthly records before installation:",
      monthly_check["before_installation"].sum())

# Network-wide monthly trend
network_trend = (
    monthly_charger.groupby("month")
    .agg(
        active_chargers=("charger_id", "nunique"),
        total_sessions=("sessions", "sum"),
        avg_capacity_utilisation=(
            "capacity_utilisation_pct", "mean"
        )
    )
    .reset_index()
)

display(network_trend.head())
display(network_trend.tail())

print("\nNetwork trend statistics:")
display(
    network_trend[
        ["total_sessions", "avg_capacity_utilisation"]
    ].describe().round(2)
)


Monthly records before installation: 1388


,month,active_chargers,total_sessions,avg_capacity_utilisation
0,2022-01,43,3394,79.281350
1,2022-02,43,3120,79.561856
2,2022-03,43,3497,79.401906
3,2022-04,43,3349,79.341963
4,2022-05,43,3497,79.426264


,month,active_chargers,total_sessions,avg_capacity_utilisation
31,2024-08,199,14037,79.538280
32,2024-09,199,13531,79.615686
33,2024-10,199,14354,79.561296
34,2024-11,199,13681,79.501691
35,2024-12,199,14249,79.527761



Network trend statistics:


,total_sessions,avg_capacity_utilisation
count,36.00,36.00
mean,8167.33,79.51
std,4432.30,0.09
min,3120.00,79.28
25%,3482.25,79.45
50%,7224.50,79.51
75%,13624.75,79.56
max,14354.00,79.71


In [13]:

# Ensure both columns are proper datetime values
sessions["start_timestamp"] = pd.to_datetime(
    sessions["start_timestamp"]
)
chargers["installation_date"] = pd.to_datetime(
    chargers["installation_date"]
)

# Attach installation dates to individual sessions
session_check = sessions.merge(
    chargers[["charger_id", "installation_date"]],
    on="charger_id",
    how="left",
    validate="many_to_one"
)

session_check["before_installation"] = (
    session_check["start_timestamp"].dt.normalize()
    < session_check["installation_date"]
)

invalid_sessions = session_check["before_installation"].sum()

print("Total sessions:", len(session_check))
print("Sessions before installation:", invalid_sessions)
print(
    "Percentage:",
    round(invalid_sessions / len(session_check) * 100, 2),
    "%"
)

print("\nExamples to investigate:")
display(
    session_check.loc[
        session_check["before_installation"],
        [
            "charger_id",
            "start_timestamp",
            "installation_date",
            "energy_kwh"
        ]
    ].head(10)
)


Total sessions: 294024
Sessions before installation: 88466
Percentage: 30.09 %

Examples to investigate:


,charger_id,start_timestamp,installation_date,energy_kwh
0,STN_001_CH_02,2022-01-01 08:30:00,2022-12-22,56.9
1,STN_001_CH_02,2022-01-01 09:20:00,2022-12-22,40.7
2,STN_001_CH_04,2022-01-01 09:00:00,2022-12-08,30.9
3,STN_001_CH_01,2022-01-01 23:20:00,2022-06-07,28.4
4,STN_001_CH_01,2022-01-01 16:10:00,2022-06-07,31.7
5,STN_001_CH_02,2022-01-01 09:30:00,2022-12-22,47.1
6,STN_001_CH_03,2022-01-01 15:20:00,2022-10-21,44.9
7,STN_001_CH_01,2022-01-01 16:10:00,2022-06-07,30.5
8,STN_001_CH_01,2022-01-01 19:40:00,2022-06-07,35.0
9,STN_001_CH_04,2022-01-01 20:10:00,2022-12-08,35.2


In [14]:

# Use monthly records with a minimum amount of activity
monthly_valid = monthly_charger[
    monthly_charger["sessions"] >= 10
].copy()

# Order records chronologically within each charger
monthly_valid = monthly_valid.sort_values(
    ["charger_id", "month"]
)

# Compare the first and last observed monthly utilisation
first_period = (
    monthly_valid.groupby("charger_id")
    .first()[["capacity_utilisation_pct"]]
    .rename(columns={
        "capacity_utilisation_pct": "first_utilisation_pct"
    })
)

last_period = (
    monthly_valid.groupby("charger_id")
    .last()[["capacity_utilisation_pct"]]
    .rename(columns={
        "capacity_utilisation_pct": "last_utilisation_pct"
    })
)

performance_change = first_period.join(last_period)

performance_change["change_percentage_points"] = (
    performance_change["last_utilisation_pct"]
    - performance_change["first_utilisation_pct"]
)

print("Chargers compared:", len(performance_change))

display(
    performance_change[
        "change_percentage_points"
    ].describe().round(3)
)

print("\nLargest decreases:")
display(
    performance_change.nsmallest(
        10, "change_percentage_points"
    ).round(2)
)


Chargers compared: 199


count    199.000
mean      -0.069
std        0.862
min       -2.026
25%       -0.696
50%       -0.002
75%        0.526
max        2.859
Name: change_percentage_points, dtype: float64


Largest decreases:


,first_utilisation_pct,last_utilisation_pct,change_percentage_points
charger_id,,,
STN_030_CH_01,80.13,78.11,-2.03
STN_027_CH_01,80.11,78.31,-1.80
STN_010_CH_03,81.09,79.29,-1.79
STN_015_CH_02,81.47,79.79,-1.68
STN_024_CH_04,80.77,79.13,-1.63
STN_023_CH_07,80.87,79.26,-1.61
STN_009_CH_04,80.14,78.61,-1.53
STN_011_CH_03,80.41,78.88,-1.53
STN_020_CH_03,81.25,79.72,-1.53


In [15]:

from scipy.stats import linregress

trend_results = []

for charger_id, group in monthly_valid.groupby("charger_id"):
    group = group.sort_values("month")

    # Require enough observations to estimate a trend
    if len(group) < 12:
        continue

    # Convert months into sequential numeric values
    x = (
        group["month"].dt.year * 12
        + group["month"].dt.month
    )
    y = group["capacity_utilisation_pct"]

    result = linregress(x, y)

    trend_results.append({
        "charger_id": charger_id,
        "months_observed": len(group),
        "slope_pp_per_month": result.slope,
        "r_squared": result.rvalue ** 2,
        "p_value": result.pvalue
    })

charger_trends = pd.DataFrame(trend_results)

print("Chargers with at least 12 monthly observations:",
      len(charger_trends))

print("\nTrend slope summary (percentage points/month):")
display(charger_trends["slope_pp_per_month"].describe().round(4))

print("\nStrongest negative trends:")
display(
    charger_trends.sort_values("slope_pp_per_month")
    .head(10)
    .round(4)
)


Chargers with at least 12 monthly observations: 199

Trend slope summary (percentage points/month):


count    199.0000
mean      -0.0017
std        0.0357
min       -0.1161
25%       -0.0169
50%       -0.0019
75%        0.0141
max        0.1300
Name: slope_pp_per_month, dtype: float64


Strongest negative trends:


,charger_id,months_observed,slope_pp_per_month,r_squared,p_value
158,STN_026_CH_05,12,-0.1161,0.2601,0.0903
164,STN_027_CH_03,12,-0.0896,0.2383,0.1074
115,STN_020_CH_03,12,-0.0803,0.2817,0.0759
137,STN_023_CH_07,12,-0.0796,0.1970,0.1483
162,STN_027_CH_01,12,-0.0741,0.1806,0.1684
124,STN_021_CH_06,12,-0.0724,0.1464,0.2196
148,STN_025_CH_03,12,-0.0704,0.4420,0.0183
157,STN_026_CH_04,12,-0.0679,0.1260,0.2576
169,STN_027_CH_08,12,-0.0655,0.1009,0.3144
179,STN_030_CH_01,12,-0.0651,0.1178,0.2748


In [16]:

# Average utilisation per charger across observed months
charger_level = (
    monthly_valid.groupby("charger_id")
    .agg(
        avg_capacity_utilisation_pct=(
            "capacity_utilisation_pct", "mean"
        ),
        monthly_utilisation_std=(
            "capacity_utilisation_pct", "std"
        ),
        months_observed=("month", "nunique"),
        total_sessions=("sessions", "sum")
    )
    .reset_index()
)

maintenance_screen = charger_level.merge(
    charger_trends,
    on=["charger_id", "months_observed"],
    how="left"
)

# Use network-relative thresholds for exploratory screening
network_utilisation = (
    maintenance_screen["avg_capacity_utilisation_pct"].median()
)

maintenance_screen["low_utilisation_flag"] = (
    maintenance_screen["avg_capacity_utilisation_pct"]
    < network_utilisation - 1
)

maintenance_screen["declining_trend_flag"] = (
    maintenance_screen["slope_pp_per_month"] < -0.05
)

maintenance_screen["screening_score"] = (
    maintenance_screen["low_utilisation_flag"].astype(int)
    + maintenance_screen["declining_trend_flag"].astype(int)
)

maintenance_screen["screening_category"] = (
    maintenance_screen["screening_score"]
    .map({
        0: "Routine Monitoring",
        1: "Review Recommended",
        2: "Higher Review Priority"
    })
)

print("Network median utilisation:",
      round(network_utilisation, 2), "%")

print("\nScreening category counts:")
display(
    maintenance_screen["screening_category"]
    .value_counts()
    .rename_axis("category")
    .reset_index(name="chargers")
)

print("\nHighest-priority review candidates:")
display(
    maintenance_screen.sort_values(
        ["screening_score", "slope_pp_per_month"],
        ascending=[False, True]
    ).head(10).round(3)
)


Network median utilisation: 79.92 %

Screening category counts:


,category,chargers
0,Routine Monitoring,146
1,Review Recommended,52
2,Higher Review Priority,1



Highest-priority review candidates:


,charger_id,avg_capacity_utilisation_pct,monthly_utilisation_std,months_observed,total_sessions,slope_pp_per_month,r_squared,p_value,low_utilisation_flag,declining_trend_flag,screening_score,screening_category
148,STN_025_CH_03,77.665,0.382,12,632,-0.070,0.442,0.018,True,True,2,Higher Review Priority
158,STN_026_CH_05,80.050,0.821,12,650,-0.116,0.260,0.090,False,True,1,Review Recommended
164,STN_027_CH_03,80.151,0.662,12,638,-0.090,0.238,0.107,False,True,1,Review Recommended
115,STN_020_CH_03,80.202,0.546,12,874,-0.080,0.282,0.076,False,True,1,Review Recommended
137,STN_023_CH_07,79.742,0.647,12,641,-0.080,0.197,0.148,False,True,1,Review Recommended
162,STN_027_CH_01,79.744,0.629,12,625,-0.074,0.181,0.168,False,True,1,Review Recommended
124,STN_021_CH_06,79.931,0.682,12,618,-0.072,0.146,0.220,False,True,1,Review Recommended
157,STN_026_CH_04,80.095,0.690,12,624,-0.068,0.126,0.258,False,True,1,Review Recommended
169,STN_027_CH_08,79.586,0.744,12,609,-0.066,0.101,0.314,False,True,1,Review Recommended
179,STN_030_CH_01,79.944,0.684,12,1333,-0.065,0.118,0.275,False,True,1,Review Recommended


In [17]:

candidate_id = "STN_025_CH_03"

# Find the candidate's rated power
candidate_power = chargers.loc[
    chargers["charger_id"] == candidate_id,
    "max_power_kW"
].iloc[0]

# Candidate's monthly performance
candidate_monthly = monthly_charger.loc[
    monthly_charger["charger_id"] == candidate_id,
    ["month", "capacity_utilisation_pct", "sessions"]
].copy()

# Same-power chargers' monthly median utilisation
peer_monthly = (
    monthly_charger.loc[
        monthly_charger["max_power_kW"] == candidate_power
    ]
    .groupby("month")["capacity_utilisation_pct"]
    .median()
    .rename("peer_median_utilisation_pct")
    .reset_index()
)

comparison = candidate_monthly.merge(
    peer_monthly,
    on="month",
    how="left"
)

comparison["difference_from_peers_pp"] = (
    comparison["capacity_utilisation_pct"]
    - comparison["peer_median_utilisation_pct"]
)

print("Candidate:", candidate_id)
print("Rated power:", candidate_power, "kW")
print("Monthly records:", len(comparison))

display(comparison.round(2))

print("\nAverage difference from same-power peers:",
      round(comparison["difference_from_peers_pp"].mean(), 2),
      "percentage points")


Candidate: STN_025_CH_03
Rated power: 300 kW
Monthly records: 12


,month,capacity_utilisation_pct,sessions,peer_median_utilisation_pct,difference_from_peers_pp
0,2024-01,78.33,46,77.64,0.69
1,2024-02,77.84,52,77.82,0.02
2,2024-03,78.10,60,77.74,0.36
3,2024-04,77.83,41,77.54,0.30
4,2024-05,77.66,53,77.75,-0.08
5,2024-06,77.72,64,77.72,-0.00
6,2024-07,77.25,46,77.68,-0.43
7,2024-08,77.45,66,77.64,-0.19
8,2024-09,77.72,44,77.83,-0.11
9,2024-10,76.84,51,77.68,-0.85



Average difference from same-power peers: -0.02 percentage points


In [18]:

# Add rated power to charger-level performance
charger_level_fixed = charger_level.merge(
    chargers[["charger_id", "max_power_kW"]],
    on="charger_id",
    how="left"
)

# Median utilisation within each rated-power group
peer_medians = (
    charger_level_fixed.groupby("max_power_kW")[
        "avg_capacity_utilisation_pct"
    ]
    .median()
    .rename("peer_median_utilisation_pct")
)

charger_level_fixed = charger_level_fixed.join(
    peer_medians,
    on="max_power_kW"
)

# Combine with each charger's trend
maintenance_screen_fixed = charger_level_fixed.merge(
    charger_trends,
    on=["charger_id", "months_observed"],
    how="left"
)

# Flag only when utilisation is >1 percentage point
# below the median for chargers of the same rated power
maintenance_screen_fixed["low_utilisation_flag"] = (
    maintenance_screen_fixed["avg_capacity_utilisation_pct"]
    < maintenance_screen_fixed["peer_median_utilisation_pct"] - 1
)

maintenance_screen_fixed["declining_trend_flag"] = (
    maintenance_screen_fixed["slope_pp_per_month"] < -0.05
)

maintenance_screen_fixed["screening_score"] = (
    maintenance_screen_fixed["low_utilisation_flag"].astype(int)
    + maintenance_screen_fixed["declining_trend_flag"].astype(int)
)

maintenance_screen_fixed["screening_category"] = (
    maintenance_screen_fixed["screening_score"].map({
        0: "Routine Monitoring",
        1: "Review Recommended",
        2: "Higher Review Priority"
    })
)

print("Screening categories after peer adjustment:")
display(
    maintenance_screen_fixed["screening_category"]
    .value_counts()
    .rename_axis("category")
    .reset_index(name="chargers")
)

print("\nPrevious candidate:")
display(
    maintenance_screen_fixed.loc[
        maintenance_screen_fixed["charger_id"] == "STN_025_CH_03",
        [
            "charger_id",
            "max_power_kW",
            "avg_capacity_utilisation_pct",
            "peer_median_utilisation_pct",
            "slope_pp_per_month",
            "low_utilisation_flag",
            "declining_trend_flag",
            "screening_category"
        ]
    ].round(3)
)


Screening categories after peer adjustment:


,category,chargers
0,Routine Monitoring,180
1,Review Recommended,19



Previous candidate:


,charger_id,max_power_kW,avg_capacity_utilisation_pct,peer_median_utilisation_pct,slope_pp_per_month,low_utilisation_flag,declining_trend_flag,screening_category
148,STN_025_CH_03,300,77.665,77.737,-0.07,False,True,Review Recommended


In [19]:

from pathlib import Path

output_dir = Path("analysis")
output_dir.mkdir(parents=True, exist_ok=True)

# Main charger-level screening table
maintenance_screen_fixed.to_csv(
    output_dir / "maintenance_screening.csv",
    index=False
)

# Monthly charger performance for trend analysis
monthly_charger.to_csv(
    output_dir / "monthly_charger_performance.csv",
    index=False
)

# Individual charger trend statistics
charger_trends.to_csv(
    output_dir / "charger_performance_trends.csv",
    index=False
)

print("Saved files:")
for file in sorted(output_dir.glob("*.csv")):
    print(file, "|", file.stat().st_size, "bytes")


Saved files:
analysis\charger_performance_trends.csv | 15835 bytes
analysis\maintenance_screening.csv | 34969 bytes
analysis\monthly_charger_performance.csv | 413601 bytes
